# Rotator - Torque Analysis

Associated tickets:
* [SITCOM-1884] Rotator torques historical/LSSTCam analysis

[SITCOM-1884]: https://rubinobs.atlassian.net/browse/SITCOM-1884

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

## Create Event Maker

We want to create a single instance of the `TMAEventMaker` object.  
Each instance might be quite heavy. 

In [ ]:
plot_path = Path("./plots_historical")
plot_path.mkdir(exist_ok=True, parents=True)

event_maker = TMAEventMaker()
efd_client = makeEfdClient()

# Data Analysis

## Single Slew Analysis

We are looking for data during slews, which is where the CCW performs longer movements.  
This comes with the first filter below.  
There are situations where `actualTorquePercentage` reports only `None` values. We need to filter these out. 

In [ ]:
day_obs = 20241208
all_events = event_maker.getEvents(day_obs)
slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

In [ ]:
evt = slew_events[0]
print(evt)

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    event=evt,
)

## Day Obs Analysis

Now we want to see the data during a whole day. 

In [ ]:
day_obs = 20241208
all_events = event_maker.getEvents(day_obs)
slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

day_obs_df = pd.DataFrame(
    columns=[
        "seq_num",
        "torque_min_0",
        "torque_avg_0",
        "torque_max_0",
        "torque_min_1",
        "torque_avg_1",
        "torque_max_1",
    ]
)

for evt in slew_events:
    df = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTRotator.motors",
        columns=[
            "torque0",
            "torque1",
        ],
        event=evt,
        warn=False,
    )

    if len(df) == 0:
        print(f"dayObs = {evt.dayObs}, seqNum = {evt.seqNum} - Empty dataframe")
        continue

    # Compute the statistics
    stats = {
        f"torque_{stat}_{i}": df[f"torque{i}"].agg(stat)
        for i in [0, 1]
        for stat in ["min", "mean", "max"]
    }
    stats["seq_num"] = evt.seqNum
    stats["max_abs_torque0"] = df["torque0"].abs().max()
    stats["max_abs_torque1"] = df["torque1"].abs().max()

    # Append the stats to the DataFrame using concat
    new_row = pd.DataFrame([stats])
    if day_obs_df.empty:
        day_obs_df = new_row  # Directly assign if day_obs_df is empty
    else:
        day_obs_df = pd.concat([day_obs_df, new_row], ignore_index=True)

In [ ]:
day_obs_df

In [ ]:
try:
    fig.clf()
    del fig
except NameError:
    pass

fig, ax = plt.subplots(num=f"day_analysis_{day_obs}")

ax.plot(
    day_obs_df["seq_num"],
    day_obs_df["torque_mean_0"],
    color="C0",
    label="Average Torque 0",
)
ax.fill_between(
    day_obs_df["seq_num"],
    day_obs_df["torque_max_0"],
    day_obs_df["torque_min_0"],
    fc="C0",
    alpha=0.3,
    label="Min/Max Torque 0",
)

ax.plot(
    day_obs_df["seq_num"],
    day_obs_df["torque_mean_1"],
    color="C1",
    label="Average Torque 1",
)
ax.fill_between(
    day_obs_df["seq_num"],
    day_obs_df["torque_max_1"],
    day_obs_df["torque_min_1"],
    fc="C1",
    alpha=0.3,
    label="Min/Max Torque 1",
)

fig.suptitle("Rotator Torques during slews")
ax.legend(loc="lower left")
ax.grid(":", alpha=0.25)
ax.set_xlabel(f"Slew ID on {day_obs}")
ax.set_ylabel(r"Torque (N$\cdot$m)", labelpad=10)
ax.set_title(f"Each datapoint is a slew event - data from {day_obs}")
plt.tight_layout()
plt.show()
fig.savefig(plot_path / f"Rotator_torques_ts_dayobs{day_obs}_ComCam.png")

In [ ]:
try:
    fig2.clf()
    del fig2
except NameError:
    pass

fig2, ax = plt.subplots(num=f"day_histogram_{day_obs}")

ax.hist(
    [day_obs_df["max_abs_torque0"], day_obs_df["max_abs_torque1"]],
    label=["Max Abs Torque 0", "Max Abs Torque 1"],
)

fig2.suptitle("Rotator Torques during slews")
ax.grid(":", alpha=0.25)
ax.set_xlabel(r"Torques N$\cdot$m")
ax.set_title(f"Histogram of the max(abs(torques)) per slew - dayobs = {day_obs}")
ax.legend()
plt.tight_layout()
plt.show()
fig2.savefig(plot_path / f"rotator_torques_hist_dayobs{day_obs}_ComCam.png")

## Campaign Analysis


Now we want to analyze the whole campaign.  
How can I make this without overcomplicating?  
I am interested in the minimum and maximum values of the torques in a day.  
Maybe I can query that directly in the EFD. Let's try it out. 

In [ ]:
day_obs_start = 20241024
day_obs_end = 20241211
fmt = "%Y%m%d"

t_start = Time(datetime.strptime(str(day_obs_start), fmt))
t_end = Time(datetime.strptime(str(day_obs_end), fmt))

query = f"""
SELECT 
    min(torque0) as min_torque_0,
    mean(torque0) as mean_torque_0,
    max(torque0) as max_torque_0,
    min(torque1) as min_torque_1,
    mean(torque1) as mean_torque_1,
    max(torque1) as max_torque_1
FROM "lsst.sal.MTRotator.motors"
WHERE time >= '{t_start.isot}Z'
AND time <= '{t_end.isot}Z'
GROUP BY time(24h)
"""

campaign_df = await efd_client.influx_client.query(query)

In [ ]:
campaign_df

In [ ]:
try:
    fig3.clf()
    del fig3
except NameError:
    pass

fig3, ax = plt.subplots(num="campaign_analysis")

ax.plot(
    campaign_df.index,
    campaign_df["mean_torque_0"],
    color="C0",
    label="Average Torque 0",
)
ax.fill_between(
    campaign_df.index,
    campaign_df["min_torque_0"],
    campaign_df["max_torque_0"],
    fc="C0",
    alpha=0.3,
    label="Min/Max Torque 0",
)

ax.plot(
    campaign_df.index,
    campaign_df["mean_torque_1"],
    color="C1",
    label="Average Torque 1",
)
ax.fill_between(
    campaign_df.index,
    campaign_df["min_torque_1"],
    campaign_df["max_torque_1"],
    fc="C1",
    alpha=0.3,
    label="Min/Max Torque 1",
)

fig3.suptitle("Rotator Torques during slews")
fig3.autofmt_xdate()

ax.legend()
ax.grid(":", alpha=0.25)
ax.set_xlabel("Day")
ax.set_ylabel(r"Torque (N$\cdot$m)")
ax.set_title(
    f"Each point is the min/max/avg in a day - data from {day_obs_start} to {day_obs_end}"
)
plt.tight_layout()
plt.show()
fig3.savefig(plot_path / f"Rotator_torques_ts_{day_obs_start}_to_{day_obs_end}.png")

In [ ]:
campaign_df["max_abs_torque_0"] = campaign_df[["max_torque_0", "min_torque_0"]].apply(
    lambda row: max(abs(row["max_torque_0"]), abs(row["min_torque_0"])), axis=1
)

campaign_df["max_abs_torque_1"] = campaign_df[["max_torque_1", "min_torque_1"]].apply(
    lambda row: max(abs(row["max_torque_1"]), abs(row["min_torque_1"])), axis=1
)

In [ ]:
try:
    fig4.clf()
    del fig4
except NameError:
    pass

fig4, ax = plt.subplots(num="histogram_campaign")

ax.hist(
    [campaign_df["max_abs_torque_0"], campaign_df["max_abs_torque_1"]],
    label=["Max Abs Torque 0", "Max Abs Torque 1"],
)

fig4.suptitle("Rotator Torques during slews")
ax.grid(":", alpha=0.25)
ax.set_ylabel(r"Torque (N$\cdot$m)")
ax.set_title(
    f"Histogram of max(abs(x)) per 24h data, from {day_obs_start} to {day_obs_end}"
)
ax.legend()
plt.show()

fig4.savefig(plot_path / f"Rotator_torques_hist_{day_obs_start}_to_{day_obs_end}.png")